# 03 — SMAP Soil Moisture (NASA)

**Owner:** Mostafa Abdelhameed
**Dataset:** SMAP L4 Global 3-hourly Surface and Root Zone Soil Moisture (`NASA/SMAP/SPL4SMGP/008`)
**Period:** 2016-01 to 2024-12
**Output:** `data/processed/smap_monthly.csv` — 108 rows, keys = `year, month`

## Why we need this

Our tool tells a farmer which crop to plant next. To do that we must know **how much water the land needs in each month**. Rain (GPM) tells us how much water comes from the sky. Soil moisture tells us how much water is actually in the ground for the roots.

## What SMAP measures

SMAP is a NASA satellite. It uses an L-band radiometer to sense water in the top soil.

Two things are important to know:

1. The instrument really sees an area of about **36 km**. The 9 km grid is made by interpolation, not by sharper sensing. Our pilot area is about 22 km x 29 km, so it fits inside roughly one real footprint. This is why we take **one value for the whole area**, not one value per 1 km cell.
2. SMAP L4 is **not a pure measurement**. It is a model (NASA Catchment Land Surface Model) with satellite data pushed into it. This matters a lot — see the conclusion.

## Plan

1. Open the SMAP L4 collection in Google Earth Engine and keep only images over our area.
2. Pick 5 useful bands:
   - `sm_surface` — water in the top 5 cm
   - `sm_rootzone` — water in the top 100 cm (**the one that matters for crops**)
   - `sm_profile` — water in the full soil column
   - `land_fraction_wilting` — how much of the land is at the wilting point (plants start to die)
   - `land_evapotranspiration_flux` — how much water leaves the soil and the plants
3. For every month from 2016 to 2024, take the **mean of all images in that month**, then take the mean over the whole area.
4. Save 108 rows to `smap_monthly.csv`.

## Why 3-hourly data is not a problem

The collection has **26,304 images** for our 9 years (8 images per day). We never download them. Earth Engine does the averaging on Google servers and sends us back one small number per month. The whole job takes less than two minutes.

## How to check the result is correct

- `n_obs` must be between **224 and 248** (8 images per day x days in the month)
- 108 rows, no missing values
- The seasonal shape must match Egypt: Egypt gets rain **only in winter**, so the soil must be driest in summer

In [1]:
import os, time
import pandas as pd, ee
from concurrent.futures import ThreadPoolExecutor

W, S, E, N = 30.70, 31.05, 30.91, 31.30
START_YEAR, END_YEAR = 2016, 2024

ee.Initialize(project="ee-fieldshift")
aoi = ee.Geometry.Rectangle([W, S, E, N])

SMAP = ee.ImageCollection("NASA/SMAP/SPL4SMGP/008").filterBounds(aoi)

# --- check first ---
print("images 2016-2024:", SMAP.filterDate("2016-01-01", "2025-01-01").size().getInfo())
print("bands:", SMAP.first().bandNames().getInfo())

images 2016-2024: 26304
bands: ['sm_surface', 'sm_rootzone', 'sm_profile', 'sm_surface_wetness', 'sm_rootzone_wetness', 'sm_profile_wetness', 'surface_temp', 'soil_temp_layer1', 'soil_temp_layer2', 'soil_temp_layer3', 'soil_temp_layer4', 'soil_temp_layer5', 'soil_temp_layer6', 'snow_mass', 'snow_depth', 'land_evapotranspiration_flux', 'overland_runoff_flux', 'baseflow_flux', 'snow_melt_flux', 'soil_water_infiltration_flux', 'land_fraction_saturated', 'land_fraction_unsaturated', 'land_fraction_wilting', 'land_fraction_snow_covered', 'heat_flux_sensible', 'heat_flux_latent', 'heat_flux_ground', 'net_downward_shortwave_flux', 'net_downward_longwave_flux', 'radiation_shortwave_downward_flux', 'radiation_longwave_absorbed_flux', 'precipitation_total_surface_flux', 'snowfall_surface_flux', 'surface_pressure', 'height_lowatmmodlay', 'temp_lowatmmodlay', 'specific_humidity_lowatmmodlay', 'windspeed_lowatmmodlay', 'vegetation_greenness_fraction', 'leaf_area_index', 'sm_rootzone_pctl', 'sm_prof

In [3]:
import os, time
import pandas as pd, ee
from concurrent.futures import ThreadPoolExecutor

# ---------- settings ----------
W, S, E, N = 30.70, 31.05, 30.91, 31.30
START_YEAR, END_YEAR = 2016, 2024
from pathlib import Path
REPO = Path.cwd()
if REPO.name == "notebooks":
    REPO = REPO.parent              # works for everyone, not just one laptop
OUT = REPO / "data" / "processed" / "smap_monthly.csv"

ee.Initialize(project="ee-fieldshift")
aoi = ee.Geometry.Rectangle([W, S, E, N])
SMAP = ee.ImageCollection("NASA/SMAP/SPL4SMGP/008").filterBounds(aoi)

BANDS = ["sm_surface", "sm_rootzone", "sm_profile",
         "land_fraction_wilting", "land_evapotranspiration_flux"]

COLS = ["year", "month",
        "sm_surface_rainfed_m3m3", "sm_rootzone_rainfed_m3m3", "sm_profile_rainfed_m3m3",
        "wilting_frac_rainfed", "et_rainfed_mm_day", "n_obs"]

# ---------- one month ----------
def smap_month(year, month):
    start = ee.Date.fromYMD(year, month, 1)
    col = SMAP.filterDate(start, start.advance(1, "month")).select(BANDS)
    stats = col.mean().reduceRegion(reducer=ee.Reducer.mean(), geometry=aoi,
                                    scale=11000, maxPixels=1e9)
    d = stats.set("n_obs", col.size()).getInfo()
    return {"year": year, "month": month,
            "sm_surface_rainfed_m3m3":  d["sm_surface"],
            "sm_rootzone_rainfed_m3m3": d["sm_rootzone"],
            "sm_profile_rainfed_m3m3":  d["sm_profile"],
            "wilting_frac_rainfed":     d["land_fraction_wilting"],
            "et_rainfed_mm_day":        d["land_evapotranspiration_flux"] * 86400,
            "n_obs":                    d["n_obs"]}

# ---------- run all months ----------
def run_one(job):
    y, m = job
    for attempt in range(3):
        try:
            return smap_month(y, m)
        except Exception as e:
            if attempt == 2:
                return {"year": y, "month": m, "error": f"{type(e).__name__}: {e}"}
            time.sleep(10)

t0 = time.time()
jobs = [(y, m) for y in range(START_YEAR, END_YEAR + 1) for m in range(1, 13)]
rows, fails = [], []

with ThreadPoolExecutor(max_workers=6) as ex:
    for i, r in enumerate(ex.map(run_one, jobs), 1):
        if "error" in r:
            fails.append(r)
            print(f"[{i}/108] {r['year']}-{r['month']:02d} FAILED {r['error']}")
        else:
            rows.append(r)
        if i % 12 == 0:
            print(f"[{i}/108] ... ({time.time() - t0:.0f}s)")

# ---------- save ----------
df = pd.DataFrame(rows)[COLS].sort_values(["year", "month"]).reset_index(drop=True)
os.makedirs(os.path.dirname(OUT), exist_ok=True)
df.to_csv(OUT, index=False)

print(f"\nsaved {len(df)} rows | {len(fails)} failures | {time.time() - t0:.0f}s")
print("file:", OUT)
print(df.columns.tolist())
print(df.groupby("month")[["wilting_frac_rainfed", "et_rainfed_mm_day"]].mean().round(3))

[12/108] ... (7s)
[24/108] ... (13s)
[36/108] ... (19s)
[48/108] ... (19s)
[60/108] ... (24s)
[72/108] ... (25s)
[84/108] ... (44s)
[96/108] ... (44s)
[108/108] ... (49s)

saved 108 rows | 0 failures | 49s
file: C:\Field-Shift-Adapting-Farms-with-NASA-Data-2026\data\processed\smap_monthly.csv
['year', 'month', 'sm_surface_rainfed_m3m3', 'sm_rootzone_rainfed_m3m3', 'sm_profile_rainfed_m3m3', 'wilting_frac_rainfed', 'et_rainfed_mm_day', 'n_obs']
       wilting_frac_rainfed  et_rainfed_mm_day
month                                         
1                     0.678              0.582
2                     0.671              0.775
3                     0.718              0.845
4                     0.818              0.559
5                     0.904              0.202
6                     0.920              0.438
7                     0.930              0.257
8                     0.949             -0.034
9                     0.945             -0.067
10                    0.929        

## Conclusion

The job finished with **108 rows and 0 failures in 49 seconds**. `n_obs` is between 224 and 248 for every month, so no month is missing images.

### The numbers were not what we expected

We expected root zone soil moisture around **0.25-0.40 m3/m3**, because our area is fully irrigated farmland. We got **0.13-0.16**. We also got:

| Month | `wilting_frac_rainfed` | `et_rainfed_mm_day` |
|---|---|---|
| January | 0.68 | 0.58 |
| May | 0.90 | 0.20 |
| August | 0.95 | **-0.03** |
| September | 0.95 | **-0.07** |

Three things here are impossible for real irrigated fields:
1. **95% of the land at wilting point** in August. Real fields in the Delta are green in August.
2. **Evapotranspiration of 0.2 mm/day in May.** Real irrigated crops in the Delta use about 5-7 mm/day.
3. **Negative evapotranspiration** in August and September. Negative means water moving *into* the soil (dew), not out of it.

### Why this happens — and why it is good news

We checked this in the science literature. **SMAP L4 does not include irrigation. By design.**

The L4 product takes only the *anomaly* information from the satellite brightness temperature and puts it into the NASA Catchment land surface model (Reichle et al., 2022). That model adds water from **rain only**. There is no irrigation in it. Published work states directly that SMAP L4 excludes irrigation effects, and that in irrigated regions the product wrongly shows dry conditions because of the coarse resolution and mixed pixels.

So all three strange numbers tell the same story: the model thinks our area is dry rainfed land. The soil has no water, so there is nothing to evaporate, so night dew makes the net flux slightly negative.

**This is not a broken dataset. It is exactly the baseline we need.**

SMAP L4 answers the question: *"What would this land look like if nobody irrigated it?"* The gap between that and reality **is the irrigation demand**.

### What we use it for

`wilting_frac_rainfed` is our **irrigation need signal**. Its seasonal shape is clean and physically correct for Egypt, where rain falls only in winter:

- **January-February: 0.67** — winter rain covers part of the water need, so a winter crop is cheaper in water
- **August-September: 0.95** — zero rain, so a summer crop needs full irrigation

This lets our tool tell a farmer not just *which* crop fits, but *how much irrigation water each option will cost in each month*.

### What we do NOT use it for

- We do **not** report `sm_surface/rootzone/profile` as the real soil moisture of the fields. They are rainfed-model values. Every column name ends in `rainfed` so nobody on the team or reading our work can misread them.
- We do **not** build any decision on `et_rainfed_mm_day`. Its values are too small and sometimes negative. We keep the column for transparency only.

### Cross-check still to do in EDA

`wilting_frac_rainfed` should move opposite to GPM rainfall (more rain -> less wilting). We will test that correlation against `gpm_monthly.csv`. If it holds, two independent NASA datasets confirm each other.